# Data Cleaning — Craigslist Used Cars Dataset
**CT118-3-3 ODL | Assignment 1 — Task 2: Data Preparation**

Dataset: [Craigslist Used Cars & Trucks (Kaggle)](https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data) —
426,880 listings x 26 columns. Reads `dataset/vehicles.csv` and writes `dataset/vehicles_cleaned.csv`
for `model_building_tuning_evaluation.ipynb`.

This notebook fixes the problems found in `eda.ipynb`. Each step says why it is done and the row
count is logged after every stage, so Section 11 shows what each decision cost.

## 1. Imports & Configuration

All the thresholds are set as constants here so the cleaning policy is easy to read and change in
one place. Paths use `os.path.join` so the notebook runs on Windows, macOS and Linux.

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 40)

# Paths
DATA_PATH       = os.path.join('dataset', 'vehicles.csv')
CLEAN_DATA_PATH = os.path.join('dataset', 'vehicles_cleaned.csv')

# Price limits. The $500 floor removes the $1 bait listings, the cap is the IQR fence from the EDA.
PRICE_MIN, PRICE_MAX = 500, 57_364

# Year and odometer limits, also the IQR fences from the EDA.
YEAR_MIN     = 1995
ODOMETER_MAX = 277_300

# What to do with missing year/odometer: 'drop' removes the rows, 'impute' median fills them.
MISSING_NUMERIC_POLICY = 'drop'

# How many model names to keep before the rest become 'other'.
TOP_N_MODELS = 500

DROP_SIZE     = True   # 72% missing
DROP_LATLONG  = True   # region and state already cover location

# Only used if posting_date cannot be parsed.
FALLBACK_SCRAPE_YEAR = 2022

# Used to find repeat listings that have no VIN.
COMPOSITE_KEY = ['manufacturer', 'model', 'year', 'odometer', 'price', 'region']

# Rows missing any of these get dropped, they are too important to guess.
CRITICAL_COLS = ['manufacturer', 'model', 'fuel', 'transmission', 'title_status']

## 2. Load Raw Data & Set Up the Cleaning Log

`log_step()` saves the row count after every stage. Section 11 turns that into one table for the
report, showing how many rows each decision removed.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
N_RAW = len(df)
print(f'Raw dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')

_log = []

def log_step(step, frame, note=''):
    '''Save the rows left after a cleaning step and print a one-line summary.'''
    prev    = _log[-1]['rows_after'] if _log else N_RAW
    removed = prev - len(frame)
    _log.append({
        'step'         : step,
        'rows_after'   : len(frame),
        'rows_removed' : removed,
        'pct_of_raw'   : round(100 * len(frame) / N_RAW, 2),
        'note'         : note,
    })
    print(f'{step:<46s} rows: {len(frame):>7,}   (removed {removed:>6,})')

log_step('0. Raw data loaded', df)

## 3. Row Validity Filters

These run before deduplication so the row kept from each duplicate group is a valid listing and
not something like a $0 repost.

### 3.1 Price — keep \$500 to \$57,364

The old filter was `price > 0`, which cleared the 32,895 zero prices but kept the \$1 and \$123
bait listings. Those are really missing prices, and since price is the target they are noise on
the label itself.

They also ruin MAPE, which divides by the true value: a \$1 listing predicted at a sensible
\$8,000 gives an error of 799,900%, so a few dozen of them make the metric meaningless. The upper
limit is the IQR fence from the EDA.

In [ ]:
n_zero  = (df['price'] == 0).sum()
n_bait  = ((df['price'] > 0) & (df['price'] < PRICE_MIN)).sum()
n_upper = (df['price'] > PRICE_MAX).sum()
print(f'{"price == 0":<34s}: {n_zero:>7,}')
print(f'{f"0 < price < ${PRICE_MIN:,}":<34s}: {n_bait:>7,}   <- kept by the old `price > 0` filter')
print(f'{f"price > ${PRICE_MAX:,} (IQR upper fence)":<34s}: {n_upper:>7,}')

df = df[(df['price'] >= PRICE_MIN) & (df['price'] <= PRICE_MAX)]
log_step(f'1. Price in [${PRICE_MIN:,}, ${PRICE_MAX:,}]', df)


### 3.2 Year and odometer — handle the NaNs first, then filter the ranges

The first version had a quiet bug: `NaN >= 1995` is False, so the range filter silently dropped
every row with a missing `year` and the median fill later never saw them. Here the missing values
are dealt with first under a stated policy and counted.

An odometer of 0 is treated as missing too. 1,965 used cars claiming zero miles is either a typo
or a seller leaving it blank, and keeping them would teach the network that zero mileage is normal.

In [ ]:
n_year_na = df['year'].isna().sum()
n_odo_na  = df['odometer'].isna().sum()
n_odo_0   = (df['odometer'] == 0).sum()
print(f'missing year            : {n_year_na:,}')
print(f'missing odometer        : {n_odo_na:,}')
print(f'odometer == 0           : {n_odo_0:,}   <- treated as missing, not as a real reading')

# Turn odometer 0 into a proper missing value before applying the policy below.
df['odometer'] = df['odometer'].replace(0, np.nan)

if MISSING_NUMERIC_POLICY == 'drop':
    df = df.dropna(subset=['year', 'odometer'])
    note = 'missing year/odometer dropped (unrecoverable)'
else:
    df['year']     = df['year'].fillna(df['year'].median())
    df['odometer'] = df['odometer'].fillna(df['odometer'].median())
    note = 'missing year/odometer median-imputed'

log_step('2. Missing year/odometer resolved', df, note)

# Now that the NaNs are gone the range filters only remove real out-of-range values.
df = df[(df['year'] >= YEAR_MIN) & (df['odometer'] <= ODOMETER_MAX)]
df['year'] = df['year'].astype(int)
log_step(f'3. year >= {YEAR_MIN}, odometer <= {ODOMETER_MAX:,}', df)

### 3.3 Rows missing a critical categorical

These five columns carry most of the non-numeric signal, so filling them with `'unknown'` would
just create a big meaningless category. Dropping them now also stops `drop_duplicates` in the next
section from treating two `NaN`s as the same car.

In [ ]:
print(df[CRITICAL_COLS].isna().sum().to_string())
df = df.dropna(subset=CRITICAL_COLS)
log_step('4. Rows missing a critical categorical', df)


## 4. Deduplication — the leakage fix

The EDA reported 0 exact duplicate rows, which is true but misleading: `id`, `url` and `image_url`
are unique per posting, so two rows always differ even when they are the same car. Dealers repost
the same vehicle across regions, and VIN `1FMJU1JT1HEA52352` shows up 261 times. The first version
of this notebook dropped `VIN` in its first step, so the duplicate check could never have found it.

The real cost is leakage. `train_test_split` shuffles rows, not cars, so copies of one vehicle end
up in both train and test — the model is scored on cars it already memorised, and nothing in the
metrics looks wrong.

About 38% of rows have no VIN, so there are two steps: rows with a VIN are deduplicated on the VIN,
and the rest on `['manufacturer', 'model', 'year', 'odometer', 'price', 'region']`, since the same
car at the same price and mileage in the same region is the same listing.

In [ ]:
# Tidy the spacing and case first, otherwise the same VIN written differently looks like two cars.
df['VIN'] = df['VIN'].astype('string').str.strip().str.upper().replace('', pd.NA)

has_vin = df['VIN'].notna()
print(f'rows with a VIN   : {has_vin.sum():,}  ({has_vin.mean()*100:.1f}%)')
print(f'rows without a VIN: {(~has_vin).sum():,}  ({(~has_vin).mean()*100:.1f}%)')

vin_counts = df.loc[has_vin, 'VIN'].value_counts()
print(f'\nDistinct VINs                     : {len(vin_counts):,}')
print(f'VINs appearing more than once     : {(vin_counts > 1).sum():,}')
print(f'Most-reposted VIN                 : {vin_counts.index[0]} x {vin_counts.iloc[0]:,}')
print('\nTop 5 most-reposted vehicles:')
print(vin_counts.head().to_string())

In [ ]:
df_vin   = df[has_vin].drop_duplicates(subset=['VIN'], keep='first')
df_novin = df[~has_vin].drop_duplicates(subset=COMPOSITE_KEY, keep='first')

print(f'VIN rows      : {has_vin.sum():>7,} -> {len(df_vin):>7,}  '
      f'({has_vin.sum() - len(df_vin):,} reposts removed)')
print(f'non-VIN rows  : {(~has_vin).sum():>7,} -> {len(df_novin):>7,}  '
      f'({(~has_vin).sum() - len(df_novin):,} duplicates removed)')

df = pd.concat([df_vin, df_novin]).sort_index()
log_step('5. Deduplicated (VIN + composite key)', df, 'removes train/test leakage')


## 5. Get the Scrape Year, then Drop Text and ID Columns

`posting_date` is used once, to get the reference year for `car_age`, and then dropped. Reading the
year from the data matters: the listings only span a few months, so using 2025 instead of the scrape
year would add the same few years to every car and distort the depreciation curve.

Dropped columns and why:

- `description` — free text with 360,911 unique values, and sellers often write the price in it.
- `posting_date` — only needed for `car_age`.
- `VIN` — its job is done in Section 4, as a feature it is just an ID.
- `id`, `url`, `region_url`, `image_url` — IDs and links, no predictive value.
- `county` — 100% missing.
- `size` — 72% missing, filling it would leave a column that is mostly one value.
- `lat`, `long` — median filling would put every affected car in the same spot in Missouri, and
  `region` and `state` already cover location.

In [ ]:
posting_year = pd.to_datetime(df['posting_date'], errors='coerce', utc=True).dt.year
if posting_year.notna().any():
    SCRAPE_YEAR = int(posting_year.max())
    print(f'Scrape year read from posting_date : {SCRAPE_YEAR} '
          f'(range {int(posting_year.min())}-{int(posting_year.max())})')
else:
    SCRAPE_YEAR = FALLBACK_SCRAPE_YEAR
    print(f'posting_date unparseable; falling back to {SCRAPE_YEAR}')

cols_to_drop = ['description', 'posting_date', 'VIN',
                'id', 'url', 'region_url', 'image_url', 'county']
if DROP_SIZE:
    cols_to_drop.append('size')
if DROP_LATLONG:
    cols_to_drop += ['lat', 'long']

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f'\nDropped {len(cols_to_drop)} columns -> {df.shape[1]} remain: {list(df.columns)}')
log_step('6. Dropped text / identifier columns', df, f'{df.shape[1]} columns remain')


## 6. `model` — Tidy the Text, Cut the Long Tail

29,667 values, and most of the tail is just free-text spelling: `f-150`, `f150`, `F-150 XLT` and
`f 150 supercrew` are the same truck. Embedding all of them would be a 29,667-row table where most
rows are seen once or twice.

So I lowercase, strip punctuation and squash the spacing to merge the variants, then keep the top
`TOP_N_MODELS` and map the rest to `other`. The coverage table below is how I picked N.

In [ ]:
df['model'] = (df['model'].astype('string')
                          .str.lower()
                          .str.strip()
                          .str.replace(r'[^a-z0-9 ]', '', regex=True)   # drop punctuation
                          .str.replace(r'\s+', ' ', regex=True)         # squash double spaces
                          .str.strip())
df['model'] = df['model'].replace('', pd.NA).fillna('unknown')

counts = df['model'].value_counts()
print(f'Distinct models after text normalisation: {counts.size:,} '
      f'(raw dataset: 29,667)\n')

print(f'{"top N":>8} {"coverage of rows":>18} {"rows -> other":>15}')
for n in [100, 250, 500, 1_000, 2_000, 5_000]:
    cov = counts.nlargest(n).sum() / len(df)
    print(f'{n:>8,} {cov*100:>17.1f}% {int((1-cov)*len(df)):>15,}')

In [ ]:
top_models  = counts.nlargest(TOP_N_MODELS).index
df['model'] = df['model'].where(df['model'].isin(top_models), 'other')

kept = (df['model'] != 'other').mean()
print(f"TOP_N_MODELS = {TOP_N_MODELS} -> {df['model'].nunique():,} categories "
      f"(incl. 'other'), covering {kept*100:.1f}% of rows directly")
print(f"\nMost common models:\n{df['model'].value_counts().head(8).to_string()}")
log_step(f'7. model normalised, top-{TOP_N_MODELS} kept', df,
         f"{df['model'].nunique():,} categories (was 29,667)")


## 7. `cylinders` — Pull Out the Number

The values are strings like `"6 cylinders"` plus an `"other"` bucket. As a category the network
cannot tell that 4 < 6 < 8, but the count is a real ordinal number and roughly tracks engine size,
so it belongs with the numeric columns.

`"other"` and missing values are not numbers, so they get a `cylinders_unknown` flag before being
median filled. That way the network can learn a separate offset for "we do not know" instead of
being told those cars have a 6-cylinder engine.

In [ ]:
raw_cyl = df['cylinders'].astype('string').str.lower()

df['cylinders_num']     = raw_cyl.str.extract(r'(\d+)', expand=False).astype(float)
df['cylinders_unknown'] = df['cylinders_num'].isna().astype(int)

n_other = raw_cyl.str.contains('other', na=False).sum()
print(f"'other' values      : {n_other:,}")
print(f'missing / unusable  : {df["cylinders_unknown"].sum():,} '
      f'({df["cylinders_unknown"].mean()*100:.1f}%) -> flagged, then median-filled')
print(f'\nExtracted distribution:\n{df["cylinders_num"].value_counts().sort_index().to_string()}')

# Median fill the flagged rows so the column is fully numeric. The flag still says which ones.
df['cylinders_num'] = df['cylinders_num'].fillna(df['cylinders_num'].median())
df = df.drop(columns=['cylinders'])
print(f'\nReplaced text `cylinders` with `cylinders_num` + `cylinders_unknown`')

## 8. Remaining Missing Values

The only columns with gaps left are `condition`, `drive`, `type` and `paint_color`. A seller
leaving those blank is itself a signal, so they become `'unknown'` as a real category instead of
being filled with the mode.

Nothing numeric needs filling here: `year` and `odometer` were handled in Section 3.2,
`cylinders_num` in Section 7, and `lat`/`long` were dropped.

In [ ]:
missing_before = df.isna().sum()
print('Columns still containing missing values:')
print(missing_before[missing_before > 0].to_string() or '  (none)')

# exclude=['number'] picks up the text columns in both pandas 2 and pandas 3.
cat_cols = df.select_dtypes(exclude=['number']).columns
df[cat_cols] = df[cat_cols].fillna('unknown').astype(str)

assert df.isna().sum().sum() == 0, 'unexpected missing values remain'
log_step('8. Categorical gaps -> "unknown"', df, 'no missing values remain')

## 9. Feature Engineering

Two things carried over from the EDA next-steps list.

**`car_age = SCRAPE_YEAR - year`** — depreciation depends on how old the car was when it was
listed, not on the model year on its own. The reference year comes from `posting_date`, and ages
are clipped at 0 because next-model-year cars do appear in the listings.

**`price_log = log1p(price)`** — price is still right-skewed after the \$57k cap, so MSE on the raw
target would be dominated by the expensive cars. `price_log` is the training target and raw `price`
is kept as well, so metrics can be reported in dollars after `expm1`.

In [ ]:
df['car_age']   = (SCRAPE_YEAR - df['year']).clip(lower=0)
df['price_log'] = np.log1p(df['price'])

print(f'car_age  (reference year {SCRAPE_YEAR}): '
      f'min {df["car_age"].min()}, median {df["car_age"].median():.0f}, max {df["car_age"].max()}')
print(f'price      skew: {df["price"].skew():>6.2f}')
print(f'log1p(price) skew: {df["price_log"].skew():>6.2f}   <- target used for training')


## 10. Final Validation

Cheap assertions so a mistake shows up here instead of quietly breaking the modelling notebook: no
missing values, every range respected, and no categorical column too large to embed.

In [ ]:
assert df.isna().sum().sum() == 0
assert df['price'].between(PRICE_MIN, PRICE_MAX).all()
assert df['year'].ge(YEAR_MIN).all()
assert df['odometer'].between(1, ODOMETER_MAX).all()
assert 'description' not in df.columns and 'VIN' not in df.columns

cat_cols = df.select_dtypes(exclude=['number']).columns
card = df[cat_cols].nunique().sort_values(ascending=False)
print('Categorical cardinality (embedding table size per column):')
print(card.to_string())
assert card.max() <= TOP_N_MODELS + 1, 'a categorical column is too high-cardinality to embed'

print(f'\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.describe().T


## 11. Cleaning Log

Every stage, what it removed, and how much of the original 426,880 listings is left at that point.

In [ ]:
log_df = pd.DataFrame(_log)
print(f'Raw rows      : {N_RAW:,}')
print(f'Cleaned rows  : {len(df):,}  ({len(df)/N_RAW*100:.1f}% retained)')
print(f'Rows removed  : {N_RAW - len(df):,}\n')
log_df


## 12. Save the Cleaned Dataset

Written to `dataset/vehicles_cleaned.csv` for `model_building_tuning_evaluation.ipynb`.

> **Scaling and encoding are left out on purpose.** `StandardScaler` and `OrdinalEncoder` learn
> means, variances and category lists from whatever data they are fitted on, so fitting them here
> would push test statistics into the training inputs — the same kind of leakage as the duplicate
> listings in Section 4, just harder to see. The modelling notebook splits 70/15/15 first and fits
> them on the training split only.

In [ ]:
os.makedirs(os.path.dirname(CLEAN_DATA_PATH) or '.', exist_ok=True)
df.to_csv(CLEAN_DATA_PATH, index=False)

size_mb = os.path.getsize(CLEAN_DATA_PATH) / 1e6
print(f'Saved {len(df):,} rows x {df.shape[1]} columns to {CLEAN_DATA_PATH} ({size_mb:,.1f} MB)')
df.head()
